# Day 40 — Bit manipulation: masks, popcount, and integers that have no width

A Python `int` behaves like a two's-complement number with infinitely many bits:
a negative number is a 1 in every position from some point upward, forever. That
splits every classic bit trick into two groups.

* **Column-by-column tricks work unchanged.** `&`, `|`, `^`, `~` look at each bit
  position on its own, so `x & -x`, `x & (x - 1)`, XOR-ing out pairs and the
  ones/twos state machine give the right answer at every width, negatives included.
* **Anything that asks "how many bits?" needs a width, and Python never stores one.**
  Popcount of a negative number, a carry that should fall off bit 31, a per-column
  count that rebuilds bit 31 - these do not crash. They return a plausible wrong
  number, or loop forever.

Each trap below sits next to the one line that pins the width.

In [1]:
from __future__ import annotations
import math, random, sys, time
from functools import reduce
from operator import xor

## 1. Python integers have no width

`bits(x, w)` shows the low `w` bits - what a `w`-bit register would hold. `u32` keeps
the low 32 bits (a C `uint32_t`), `s32` reads those 32 bits back as signed: bit 31 is
worth -2**31 instead of +2**31. Those two functions are the whole fix for every trap
in this notebook.

![img](imgs/day40_1.png)

In [2]:
W32 = 32


M32 = (1 << 32) - 1            # 0xFFFFFFFF: the 32 bits a C int would keep


# ------------------------------------------------------------------ 1. width
def bits(x: int, width: int = 8) -> str:
    """The low `width` bits of x as a string - what a width-bit register holds."""
    return format(x & ((1 << width) - 1), f'0{width}b')


def u32(x: int) -> int:
    """Keep the low 32 bits: the unsigned value a C uint32_t would hold."""
    return x & M32


def s32(u: int) -> int:
    """Read 32 bits as a signed int: bit 31 is worth -2**31, not +2**31."""
    u &= M32
    return u - (1 << 32) if u >> 31 else u


for x in (5, -5):
    print(f'{x:>3}: bin() -> {bin(x):<7}  low 8 bits {bits(x)}  low 32 bits {bits(x, 32)}')
print('~5 =', ~5, '   -5 >> 1 =', -5 >> 1)
print(f'u32(-5) = {u32(-5):,}   s32(u32(-5)) = {s32(u32(-5))}   s32(0x80000000) = {s32(0x80000000):,}')

  5: bin() -> 0b101    low 8 bits 00000101  low 32 bits 00000000000000000000000000000101
 -5: bin() -> -0b101   low 8 bits 11111011  low 32 bits 11111111111111111111111111111011
~5 = -6    -5 >> 1 = -3
u32(-5) = 4,294,967,291   s32(u32(-5)) = -5   s32(0x80000000) = -2,147,483,648


## 2. The toolkit on x = 180 = 0b10110100

Setting, clearing, toggling and testing one bit are shifts plus one operator. The
three worth memorising work on the *lowest* bit, because subtracting 1 turns the
lowest 1 into 0 and every 0 below it into 1, and `-x` is `~x + 1`:

* `x & (x - 1)` drops the lowest 1,
* `x & -x` keeps only the lowest 1 (the Fenwick tree step, and how Day 39's bitmask
  N-Queens picked its next column),
* `x | (x + 1)` turns on the lowest 0.

![img](imgs/day40_2.png)

In [3]:
# ---------------------------------------------------------------- 2. toolkit
def test_bit(x: int, k: int) -> int:
    return (x >> k) & 1


def set_bit(x: int, k: int) -> int:
    return x | (1 << k)


def clear_bit(x: int, k: int) -> int:
    return x & ~(1 << k)


def toggle_bit(x: int, k: int) -> int:
    return x ^ (1 << k)


def lowbit(x: int) -> int:
    """The lowest set bit alone (Fenwick trees, Day 39's bitmask N-Queens)."""
    return x & -x


def clear_lowest(x: int) -> int:
    """x with its lowest set bit turned off."""
    return x & (x - 1)


def set_lowest_zero(x: int) -> int:
    """x with its lowest 0 bit turned on."""
    return x | (x + 1)


def trailing_zeros(x: int) -> int:
    """How many 0s sit below the lowest 1 (x > 0)."""
    return (x & -x).bit_length() - 1


def gray(i: int) -> int:
    """Gray code: consecutive values differ in exactly one bit."""
    return i ^ (i >> 1)


TOOLKIT = [
    ('x >> 2 & 1',   'test bit 2',                lambda x: test_bit(x, 2)),
    ('x | 1 << 3',   'set bit 3',                 lambda x: set_bit(x, 3)),
    ('x & ~(1 << 4)', 'clear bit 4',              lambda x: clear_bit(x, 4)),
    ('x ^ 1 << 7',   'toggle bit 7',              lambda x: toggle_bit(x, 7)),
    ('x & -x',       'keep only the lowest 1',    lowbit),
    ('x & (x - 1)',  'drop the lowest 1',         clear_lowest),
    ('x | (x + 1)',  'turn on the lowest 0',      set_lowest_zero),
    ('x ^ (x >> 1)', 'Gray code of x',            gray),
]


x = 180
print(f'{"expression":<16}{"meaning":<26}{"bits":>10}{"value":>7}')
for expr, meaning, fn in TOOLKIT:
    v = fn(x)
    print(f'{expr:<16}{meaning:<26}{bits(v):>10}{v:>7}')
print('trailing zeros:', trailing_zeros(x), '  Gray codes 0..7:', [bits(gray(i), 3) for i in range(8)])

expression      meaning                         bits  value
x >> 2 & 1      test bit 2                  00000001      1
x | 1 << 3      set bit 3                   10111100    188
x & ~(1 << 4)   clear bit 4                 10100100    164
x ^ 1 << 7      toggle bit 7                00110100     52
x & -x          keep only the lowest 1      00000100      4
x & (x - 1)     drop the lowest 1           10110000    176
x | (x + 1)     turn on the lowest 0        10110101    181
x ^ (x >> 1)    Gray code of x              11101110    238
trailing zeros: 2   Gray codes 0..7: ['000', '001', '011', '010', '110', '111', '101', '100']


**Precedence trap.** Arithmetic binds tighter than `<<`, and `<<` tighter than `&`,
`^`, `|`. So `1 << n - 1` is `1 << (n - 1)` - a single bit - not `n` ones.

In [4]:
def full_mask_buggy(n: int) -> int:
    """Meant to be n ones. `-` binds tighter than `<<`, so this is 1 << (n - 1)."""
    return 1 << n - 1


def full_mask(n: int) -> int:
    return (1 << n) - 1


for n in (4, 8):
    print(f'n = {n}: 1 << n - 1 = {full_mask_buggy(n)}   (1 << n) - 1 = {full_mask(n)}')

n = 4: 1 << n - 1 = 8   (1 << n) - 1 = 15
n = 8: 1 << n - 1 = 128   (1 << n) - 1 = 255


## 3. popcount five ways

The shift loop looks at every bit up to the top 1. Kernighan's loop runs once per 1,
because each `x &= x - 1` removes exactly one. SWAR ("SIMD within a register") adds
all bits in parallel inside one word: 16 two-bit counts, then 8 four-bit counts, then
4 byte counts, then one multiply sums the bytes.

![img](imgs/day40_3.png)

In [5]:
# --------------------------------------------------------------- 3. popcount
def popcount_bin(x: int) -> int:
    return bin(x).count('1')


def popcount_builtin(x: int) -> int:
    return x.bit_count()                  # Python 3.10+


def popcount_shift(x: int) -> int:
    """Look at every bit: the loop runs bit_length(x) times."""
    count = 0
    while x:
        count += x & 1
        x >>= 1
    return count


def popcount_kernighan(x: int) -> int:
    """Each x &= x - 1 removes the lowest 1: the loop runs popcount(x) times."""
    count = 0
    while x:
        x &= x - 1
        count += 1
    return count


def popcount_swar32(x: int) -> int:
    """Add the bits in parallel inside one word: pairs, then nibbles, then bytes."""
    x = x - ((x >> 1) & 0x55555555)                   # 16 two-bit counts
    x = (x & 0x33333333) + ((x >> 2) & 0x33333333)    # 8 four-bit counts
    x = (x + (x >> 4)) & 0x0F0F0F0F                   # 4 byte counts
    return ((x * 0x01010101) & M32) >> 24             # sum of the 4 bytes


BYTE_TABLE = [bin(i).count('1') for i in range(256)]


def popcount_table32(x: int) -> int:
    return (BYTE_TABLE[x & 255] + BYTE_TABLE[(x >> 8) & 255]
            + BYTE_TABLE[(x >> 16) & 255] + BYTE_TABLE[(x >> 24) & 255])


def popcount32(x: int) -> int:
    """What C's __builtin_popcount / Java's Integer.bitCount return, negatives too."""
    return (x & M32).bit_count()


POPCOUNTS = [('bin(x).count("1")', popcount_bin),
             ('x.bit_count()', popcount_builtin),
             ('shift loop', popcount_shift),
             ("Kernighan x &= x-1", popcount_kernighan),
             ('SWAR (32-bit)', popcount_swar32),
             ('byte table', popcount_table32)]


def swar_stages(x: int) -> list[int]:
    """The word after each SWAR step, for the figure and the demo."""
    a = x - ((x >> 1) & 0x55555555)
    b = (a & 0x33333333) + ((a >> 2) & 0x33333333)
    c = (b + (b >> 4)) & 0x0F0F0F0F
    return [x, a, b, c, ((c * 0x01010101) & M32) >> 24]


def kernighan_trace(x: int, cap: int = 64) -> list[int]:
    """Every value x takes under x &= x - 1, stopping at 0 or after `cap` steps."""
    out = [x]
    while x and len(out) <= cap:
        x &= x - 1
        out.append(x)
    return out


def best_time(fn, repeat: int = 5) -> float:
    best = float('inf')
    for _ in range(repeat):
        t = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t)
    return best

In C, SWAR is the fast one. In Python it is not: every `&`, `>>` and `+` is a trip
through the interpreter, so the method with the fewest Python-level operations wins,
and that is the builtin `int.bit_count()` (Python 3.10+), which runs the loop in C.

![img](imgs/day40_4.png)

In [6]:
rng = random.Random(40)
xs = [rng.getrandbits(32) for _ in range(50_000)]
ref = [v.bit_count() for v in xs]
for name, fn in POPCOUNTS:
    assert [fn(v) for v in xs] == ref, name
    t = best_time(lambda: [fn(v) for v in xs], repeat=3)
    print(f'{name:<22}{t * 1000:>7.1f} ms')
print('Kernighan trace of 180:', kernighan_trace(180))
print('SWAR stages of 180:', [bits(s) for s in swar_stages(180)[:4]], '->', swar_stages(180)[4])

bin(x).count("1")        22.0 ms
x.bit_count()             5.4 ms
shift loop              166.4 ms
Kernighan x &= x-1       63.2 ms
SWAR (32-bit)            23.7 ms
byte table               19.4 ms
Kernighan trace of 180: [180, 176, 160, 128, 0]
SWAR stages of 180: ['10110100', '01100100', '00110001', '00000100'] -> 4


## 4. popcount of a negative number

C and Java say -5 has 31 ones (in 32 bits). Python has no 32, so `bin(-5)` is
`'-0b101'` and `(-5).bit_count()` is documented as the popcount of `abs(x)`: both say 2.
Kernighan's loop never ends, because `x & (x - 1)` of a negative number is still
negative - the 1s above never run out.

In [7]:
print('C / Java answer:', popcount32(-5))
print('bin(-5).count("1") =', popcount_bin(-5), '  (-5).bit_count() =', popcount_builtin(-5))
tr = kernighan_trace(-5, cap=40)
print('Kernighan, first values:', tr[:6], f'... after 40 rounds: {tr[-1]:,}')
print('fix: (-5 & M32).bit_count() =', (-5 & M32).bit_count())

C / Java answer: 31
bin(-5).count("1") = 2   (-5).bit_count() = 2
Kernighan, first values: [-5, -6, -8, -16, -32, -64] ... after 40 rounds: -2,199,023,255,552
fix: (-5 & M32).bit_count() = 31


## 5. LeetCode 338 - Counting Bits

Popcount of every `i` in `0..n` in one pass: `i >> 1` is `i` without its last bit and
is already in the table, so `ans[i] = ans[i >> 1] + (i & 1)`. Dropping the lowest 1
instead gives the same table: `ans[i] = ans[i & (i - 1)] + 1`.

In [8]:
# ------------------------------------------------------------ 5. LeetCode 338
class Solution338:
    def countBits(self, n: int) -> list[int]:
        ans = [0] * (n + 1)
        for i in range(1, n + 1):
            ans[i] = ans[i >> 1] + (i & 1)      # i without its last bit, plus that bit
        return ans


def count_bits_lowbit(n: int) -> list[int]:
    """Same table, other recurrence: i has one more 1 than i with its lowest 1 dropped."""
    ans = [0] * (n + 1)
    for i in range(1, n + 1):
        ans[i] = ans[i & (i - 1)] + 1
    return ans


print(Solution338().countBits(5))
print(Solution338().countBits(16))
assert Solution338().countBits(2000) == count_bits_lowbit(2000)

[0, 1, 1, 2, 1, 2]
[0, 1, 1, 2, 1, 2, 2, 3, 1, 2, 2, 3, 2, 3, 3, 4, 1]


## 6. A Python int as a set of a million members

Bit `v` set means "v is in the set". Intersection is `&`, union `|`, size
`bit_count()` - each one a single C loop over machine words. The trap is building it:
`x |= 1 << v` creates a brand-new big integer every time, so adding `m` members to a
million-bit set copies about `m` million bits. Fill a `bytearray` and convert once.

In [9]:
# ------------------------------------------------------------- 6. bitsets
def to_bitset(values, universe: int) -> int:
    """Set bit v for every v: fill a bytearray, convert once. Linear time."""
    buf = bytearray((universe + 7) // 8)
    for v in values:
        buf[v >> 3] |= 1 << (v & 7)
    return int.from_bytes(buf, 'little')


def to_bitset_slow(values) -> int:
    """Looks the same, is quadratic: every |= builds a brand-new big integer."""
    x = 0
    for v in values:
        x |= 1 << v
    return x


def members(x: int) -> list[int]:
    """Indices of the set bits, lowest first (fine for small or sparse x)."""
    out = []
    while x:
        low = x & -x
        out.append(low.bit_length() - 1)
        x ^= low
    return out


def jaccard_bits(a: int, b: int) -> float:
    return (a & b).bit_count() / (a | b).bit_count()


rng = random.Random(40)
U = 1_000_000
A = set(rng.sample(range(U), 100_000)); B = set(rng.sample(range(U), 100_000))
a, b = to_bitset(A, U), to_bitset(B, U)
t_set = best_time(lambda: len(A & B), repeat=10)
t_bit = best_time(lambda: (a & b).bit_count(), repeat=10)
print(f'|A & B| = {len(A & B):,} = {(a & b).bit_count():,}')
print(f'set {t_set * 1000:.2f} ms   bits {t_bit * 1000:.3f} ms   ({t_set / t_bit:.0f}x)')
print(f'memory: set {sys.getsizeof(A):,} B   int {sys.getsizeof(a):,} B')
few = rng.sample(range(U), 20_000)
t_slow = best_time(lambda: to_bitset_slow(few), repeat=1)
t_fast = best_time(lambda: to_bitset(few, U), repeat=3)
print(f'build 20,000 members: x |= 1 << v {t_slow * 1000:.0f} ms, bytearray {t_fast * 1000:.1f} ms')

|A & B| = 9,955 = 9,955
set 7.42 ms   bits 0.122 ms   (61x)
memory: set 4,194,520 B   int 133,360 B
build 20,000 members: x |= 1 << v 376 ms, bytearray 3.6 ms


## 7. Gosper's hack - every k-subset in increasing order

The next larger number with the same count of 1s: add the lowest 1 so it carries into
the block of 1s above it, then put the 1s that the carry swallowed back at the bottom.

In [10]:
# ------------------------------------------------------------- 7. Gosper
def next_same_popcount(x: int) -> int:
    """Gosper's hack: the next larger integer with the same number of 1s."""
    c = x & -x                   # lowest 1
    r = x + c                    # carry it into the block of 1s above it
    return (((r ^ x) >> 2) // c) | r   # put the leftover 1s back at the bottom


def k_subsets(n: int, k: int) -> list[int]:
    """Every n-bit mask with exactly k ones, in increasing order."""
    if k == 0:
        return [0]
    out, x = [], (1 << k) - 1
    while x < 1 << n:
        out.append(x)
        x = next_same_popcount(x)
    return out


print([bits(m, 5) for m in k_subsets(5, 3)])
print(f'n = 20, k = 3: {len(k_subsets(20, 3)):,} masks instead of filtering {1 << 20:,}')

['00111', '01011', '01101', '01110', '10011', '10101', '10110', '11001', '11010', '11100']
n = 20, k = 3: 1,140 masks instead of filtering 1,048,576


## 8. LeetCode 136 / 137 - what XOR can and cannot cancel

136: XOR is its own inverse, so pairs cancel column by column and the single number
remains - negatives included, no width needed.

137 (everything appears three times): XOR cancels pairs, not triples. The count-per-column
answer sums each of 32 columns mod 3 - and needs `s32` at the end, or bit 31 comes back
worth +2**31. The ones/twos state machine keeps a mod-3 counter per column using only
`^`, `&`, `~`, so it needs no width at all.

![img](imgs/day40_6.png)

In [11]:
# ------------------------------------------------------- 8. LeetCode 136 / 137
class Solution136:
    def singleNumber(self, nums: list[int]) -> int:
        return reduce(xor, nums, 0)             # pairs cancel, column by column


class Solution137:
    def singleNumber(self, nums: list[int]) -> int:
        ones = twos = 0                         # per column: seen once / seen twice (mod 3)
        for x in nums:
            ones = (ones ^ x) & ~twos
            twos = (twos ^ x) & ~ones
        return ones


def single_number_ii_count(nums: list[int]) -> int:
    """Count each of 32 columns mod 3 - then read bit 31 as the sign."""
    res = 0
    for k in range(32):
        if sum((x >> k) & 1 for x in nums) % 3:
            res |= 1 << k
    return s32(res)


def single_number_ii_count_nosign(nums: list[int]) -> int:
    """Same loop, last line forgotten: bit 31 comes back worth +2**31."""
    res = 0
    for k in range(32):
        if sum((x >> k) & 1 for x in nums) % 3:
            res |= 1 << k
    return res


print(Solution136().singleNumber([4, 1, 2, 1, 2]), Solution136().singleNumber([-7, 3, 3]))
for nums in ([2, 2, 3, 2], [0, 1, 0, 1, 0, 1, 99], [-2, -2, -3, -2]):
    print(nums, Solution137().singleNumber(nums), single_number_ii_count(nums),
          single_number_ii_count_nosign(nums))

4 -7
[2, 2, 3, 2] 3 3 3
[0, 1, 0, 1, 0, 1, 99] 99 99 99
[-2, -2, -3, -2] -3 -3 4294967293


## 9. LeetCode 371 - Sum of Two Integers without + or -

`a ^ b` is the sum without carries, `(a & b) << 1` is the carry. Repeat until the carry
is 0. In C the carry eventually shifts past bit 31 and disappears. In Python there is
no bit 31 to fall off: for `(-1, 1)` the pair becomes `(-2, 2)`, `(-4, 4)`, ... forever.
Masking to 32 bits makes it stop; reading bit 31 as the sign makes the answer right.

![img](imgs/day40_5.png)

In [12]:
# ------------------------------------------------------------ 9. LeetCode 371
def get_sum_naive(a: int, b: int, cap: int | None = None):
    """The C answer typed into Python. Returns None if `cap` rounds pass."""
    rounds = 0
    while b:
        if cap is not None and rounds == cap:
            return None
        a, b = a ^ b, (a & b) << 1          # sum without carry, carry moved up
        rounds += 1
    return a


def get_sum_trace(a: int, b: int, cap: int = 40, mask: bool = False) -> list[tuple[int, int]]:
    out = [(a, b)]
    if mask:
        a, b = a & M32, b & M32
        out = [(a, b)]
    while b and len(out) <= cap:
        a, b = a ^ b, (a & b) << 1
        if mask:
            a, b = a & M32, b & M32
        out.append((a, b))
    return out


def get_sum_masked(a: int, b: int) -> int:
    """32-bit wrap added - but the answer is still read as unsigned."""
    a, b = a & M32, b & M32
    while b:
        a, b = (a ^ b) & M32, ((a & b) << 1) & M32
    return a


class Solution371:
    def getSum(self, a: int, b: int) -> int:
        a, b = a & M32, b & M32
        while b:
            a, b = (a ^ b) & M32, ((a & b) << 1) & M32   # the carry falls off bit 31
        return a if a < 1 << 31 else a - (1 << 32)       # read bit 31 as the sign


for a_, b_ in ((1, 2), (2, 3), (-1, -1), (-1, 1)):
    print(f'{a_:>2} + {b_:>2}: naive {get_sum_naive(a_, b_, cap=100)}  '
          f'masked {get_sum_masked(a_, b_):,}  fixed {Solution371().getSum(a_, b_)}')
print('naive (-1, 1) trace:', get_sum_trace(-1, 1, cap=4))

 1 +  2: naive 3  masked 3  fixed 3
 2 +  3: naive 5  masked 5  fixed 5
-1 + -1: naive -2  masked 4,294,967,294  fixed -2
-1 +  1: naive None  masked 0  fixed 0
naive (-1, 1) trace: [(-1, 1), (-2, 2), (-4, 4), (-8, 8), (-16, 16)]


## 10. LeetCode 191 / 231

191 is Kernighan's loop on an unsigned 32-bit input. 231 is `n & (n - 1) == 0` - a
power of two has exactly one 1 - but only with `n > 0` in front: 0 has no 1 to drop
either, and the buggy version still passes the examples 1, 16 and 3.

In [13]:
# ------------------------------------------------------- 10. LeetCode 191 / 231
class Solution191:
    def hammingWeight(self, n: int) -> int:
        count = 0
        while n:
            n &= n - 1
            count += 1
        return count


class Solution231:
    def isPowerOfTwo(self, n: int) -> bool:
        return n > 0 and n & (n - 1) == 0


def is_power_of_two_buggy(n: int) -> bool:
    return n & (n - 1) == 0                     # 0 has no 1 to drop either


print([Solution191().hammingWeight(v) for v in (11, 128, 4294967293)])
print([(v, Solution231().isPowerOfTwo(v), is_power_of_two_buggy(v)) for v in (1, 16, 3, 0)])

[3, 1, 31]
[(1, True, True), (16, True, True), (3, False, False), (0, False, True)]


## Tests

In [14]:
def run_tests() -> None:
    assert bits(-5) == '11111011' and u32(-5) == 4294967291 and s32(u32(-5)) == -5
    assert ~5 == -6 and -5 >> 1 == -3 and s32(0x80000000) == -2**31
    x = 180
    assert [fn(x) for _, _, fn in TOOLKIT] == [1, 188, 164, 52, 4, 176, 181, 238]
    assert trailing_zeros(180) == 2 and full_mask_buggy(4) == 8 and full_mask(4) == 15
    assert all(bin(gray(i) ^ gray(i + 1)).count('1') == 1 for i in range(1000))
    rng = random.Random(1)
    for _ in range(3000):
        v = rng.getrandbits(32)
        want = bin(v).count('1')
        assert (popcount_builtin(v) == popcount_shift(v) == popcount_kernighan(v)
                == popcount_swar32(v) == popcount_table32(v) == want)
    assert swar_stages(180)[-1] == 4
    assert popcount_bin(-5) == popcount_builtin(-5) == 2 and popcount32(-5) == 31
    tr = kernighan_trace(-5, cap=40)
    assert len(tr) == 41 and tr[-1] != 0
    assert Solution338().countBits(5) == [0, 1, 1, 2, 1, 2]
    assert Solution338().countBits(5000) == count_bits_lowbit(5000) == [bin(i).count('1') for i in range(5001)]
    vals = rng.sample(range(10_000), 500)
    assert to_bitset(vals, 10_000) == to_bitset_slow(vals) and members(to_bitset(vals, 10_000)) == sorted(vals)
    for n in range(1, 13):
        for k in range(0, n + 1):
            ks = k_subsets(n, k)
            assert len(ks) == math.comb(n, k) and ks == sorted(ks)
            assert all(m.bit_count() == k and m < 1 << n for m in ks)
    assert Solution136().singleNumber([4, 1, 2, 1, 2]) == 4 and Solution136().singleNumber([-7, 3, 3]) == -7
    for nums, want in (([2, 2, 3, 2], 3), ([0, 1, 0, 1, 0, 1, 99], 99), ([-2, -2, -3, -2], -3)):
        assert Solution137().singleNumber(nums) == single_number_ii_count(nums) == want
    assert single_number_ii_count_nosign([2, 2, 3, 2]) == 3
    assert single_number_ii_count_nosign([0, 1, 0, 1, 0, 1, 99]) == 99
    assert single_number_ii_count_nosign([-2, -2, -3, -2]) == 4294967293
    for a, b in ((1, 2), (2, 3), (-1, -1), (-1, 1), (-1000, 999), (2**31 - 1, -2**31), (0, 0)):
        assert Solution371().getSum(a, b) == a + b
    assert get_sum_naive(1, 2) == 3 and get_sum_naive(2, 3) == 5 and get_sum_naive(-1, 1, cap=1000) is None
    assert get_sum_masked(-1, -1) == 4294967294 and get_sum_masked(-1, 1) == 0
    assert len(get_sum_trace(-1, 1, cap=100, mask=True)) - 1 == 32
    assert [Solution191().hammingWeight(v) for v in (11, 128, 4294967293)] == [3, 1, 31]
    assert [Solution231().isPowerOfTwo(v) for v in (1, 16, 3, 0, -16)] == [True, True, False, False, False]
    assert is_power_of_two_buggy(0) is True


run_tests()
print('all assertions passed')

all assertions passed
